In [3]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.xls', encoding='utf-8-sig')

In [5]:
print("행/열:", df.shape)

행/열: (1470, 35)


In [ ]:
print(df.dtypes)
print("결측치 총합", df.isnull().sum().sum())

Age                          int64
Attrition                   object
BusinessTravel              object
DailyRate                    int64
Department                  object
DistanceFromHome             int64
Education                    int64
EducationField              object
EmployeeCount                int64
EmployeeNumber               int64
EnvironmentSatisfaction      int64
Gender                      object
HourlyRate                   int64
JobInvolvement               int64
JobLevel                     int64
JobRole                     object
JobSatisfaction              int64
MaritalStatus               object
MonthlyIncome                int64
MonthlyRate                  int64
NumCompaniesWorked           int64
Over18                      object
OverTime                    object
PercentSalaryHike            int64
PerformanceRating            int64
RelationshipSatisfaction     int64
StandardHours                int64
StockOptionLevel             int64
TotalWorkingYears   

In [7]:
print(df['Attrition'].value_counts())
print(df['Attrition'].value_counts(normalize=True).round(3) * 100)

Attrition
No     1233
Yes     237
Name: count, dtype: int64
Attrition
No     83.9
Yes    16.1
Name: proportion, dtype: float64


In [8]:
ot_yes = df[df['OverTime'] == 'Yes']['JobSatisfaction']
ot_no = df[df['OverTime'] == 'No']['JobSatisfaction']

print("야근 O 평균 만족도:", round(ot_yes.mean(), 3))
print("야근 X 평균 만족도:", round(ot_no.mean(), 3))
print("야근 O 인원:", len(ot_yes), "/ 야근 X 인원:", len(ot_no))

야근 O 평균 만족도: 2.772
야근 X 평균 만족도: 2.712
야근 O 인원: 416 / 야근 X 인원: 1054


In [11]:
# 1~4 순서형 변수 -> 비모수 검정
u_stat, p_value = stats.mannwhitneyu(ot_yes, ot_no, alternative='two-sided')
print(f"Mann-Whitney U: statistic={u_stat}, p-value={p_value:.4f}")

t_stat, t_p = stats.ttest_ind(ot_yes, ot_no)
print(f"t-test (참고용): statistic={t_stat:.4f}, p-value={t_p:.4f}")

Mann-Whitney U: statistic=226952.0, p-value=0.2743
t-test (참고용): statistic=0.9405, p-value=0.3471


In [12]:
crosstab = pd.crosstab(df['OverTime'], df['Attrition'], normalize='index') * 100
print(crosstab.round(1))

chi2, p, dof, expected = stats.chi2_contingency(pd.crosstab(df['OverTime'], df['Attrition']))
print(f"\nChi-square: {chi2:.4f}, p-value: {p:.4f}, dof={dof}")

Attrition    No   Yes
OverTime             
No         89.6  10.4
Yes        69.5  30.5

Chi-square: 87.5643, p-value: 0.0000, dof=1


In [ ]:
# 부서별 야근 비율
dept_ot = df.groupby('Department')['OverTime'].apply(lambda x: (x == 'Yes').mean() * 100)
print(dept_ot.round(1))

Department
Human Resources           27.0
Research & Development    28.2
Sales                     28.7
Name: OverTime, dtype: float64


#### insight

야근 안 하는 그룹: 퇴사율 10.4%
야근 하는 그룹: 퇴사율 30.5% (약 3배)
부서별 야근 비율: HR 27.0% / R&D 28.2% / Sales 28.7%